# StyleGAN — Style-Based Generator Architecture

This notebook implements a simplified **StyleGAN** (Karras et al., 2019), following Module 18.

Earlier GANs in this series take a noise vector `z` and let the Generator transform it into an image. The latent is consumed at the input; everything downstream is "just feature extraction."

StyleGAN takes a fundamentally different view. Instead of feeding `z` into the synthesis network, it first passes `z` through a **mapping network** to produce an intermediate latent `w`:

```
z -> mapping network -> w -> styles injected at every layer -> image
```

Three architectural ideas drive the difference:

1. **Mapping network** (`z -> w`): a small MLP that learns to "unwind" the random noise distribution into a smoother, more disentangled intermediate space.
2. **Learned constant input**: the synthesis network starts from a *learned* 4x4 feature map, not from the latent. All variation comes from styles injected later.
3. **Style injection via AdaIN**: at every synthesis layer, the activation statistics are replaced by values derived from `w`. Coarse layers absorb coarse styles (pose, shape); fine layers absorb fine styles (texture, micro-details).

We also add **per-layer noise injection** so the model has a separate channel for *stochastic fine detail* (skin texture, hair-level randomness) that doesn't have to be encoded in `w`.

We skip the original StyleGAN's progressive growing (covered in Module 17) and train at a fixed 32x32 throughout. The architectural innovations of StyleGAN are *orthogonal* to progressive growing; combining them is what the original paper does, but the style-based ideas stand on their own.

## Implementation Plan

- **Mapping network**: 4-layer MLP from `z in R^100` to `w in R^128`. LeakyReLU activations.
- **Synthesis network**: starts from a *learned* `4x4` constant, then four style-controlled blocks at resolutions `4`, `8`, `16`, `32`. Each block does `Upsample -> Conv -> Noise -> AdaIN -> LeakyReLU`. Channel count decreases as resolution increases.
- **AdaIN**: per-channel instance normalization followed by a learned affine transform of the style vector. `y_s` and `y_b` come from `w` via a small linear layer.
- **Noise injection**: a single-channel per-pixel Gaussian noise broadcast across channels with a learnable per-channel weight.
- **Discriminator**: standard PatchGAN (4-layer, output `4x4` logit map for 32x32 input). Same recipe as Pix2Pix / CycleGAN.
- **Loss**: `BCEWithLogitsLoss` on the patch map, standard adversarial training.
- **Style mixing demo**: a cell at the end that swaps `w` vectors between two latents at a chosen layer, so you can visually see coarse/middle/fine control.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
import numpy as np
import os

torch.manual_seed(42)
np.random.seed(42)

## 1. Setup and Hyperparameters

StyleGAN has *more* parameters than DCGAN at the same image size, mostly because every synthesis layer carries its own AdaIN affine transform. We drop `batch_size` to compensate.

In [ ]:
latent_dim = 100    # z space
w_dim      = 128    # w space (intermediate latent)
img_channels = 1
img_size     = 32   # fixed resolution (skip progressive growing for clarity)
features_g   = 64
features_d   = 64
batch_size   = 16
lr           = 2e-4
betas        = (0.5, 0.999)
epochs       = 25

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

os.makedirs('samples_StyleGAN', exist_ok=True)

## 2. Data — MNIST at 32x32

Same MNIST pipeline as before, sized to `32x32`.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

dataloader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=True, download=True, transform=transform),
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)

## 3. Weight Initialization

Same DCGAN-style init. StyleGAN's authors also rescale the initial weights of the mapping network by a small factor; we keep the simpler default.

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)
    elif classname.find('Linear') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)

## 4. The Mapping Network `z -> w`

A small MLP that turns the original random latent `z` into the intermediate latent `w`. We use 4 fully connected layers with LeakyReLU. The mapping is the *only* place where randomness enters; everything downstream in the synthesis network is a deterministic function of `w` plus injected noise.

In [ ]:
class MappingNetwork(nn.Module):
    def __init__(self, latent_dim=100, w_dim=128, n_layers=4):
        super().__init__()
        layers = []
        in_dim = latent_dim
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, w_dim))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            in_dim = w_dim
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)

## 5. AdaIN — Adaptive Instance Normalization

AdaIN is the mechanism that injects style into a feature map. The math is two steps:

1. **Normalize** each feature channel to zero mean, unit variance (instance norm).
2. **Affine-rescale** using a per-channel `scale` and `bias` derived from `w` via a small linear layer.

Formally:

```
AdaIN(x, w) = y_s * (x - mean(x)) / std(x)  +  y_b
```

where `y_s, y_b` are linear projections of `w`. Different synthesis layers each have their own AdaIN, so each layer gets its own style slice.

In [ ]:
class AdaIN(nn.Module):
    def __init__(self, w_dim, channels):
        super().__init__()
        self.channels = channels
        # Linear from w -> (scale, bias) for every channel
        self.fc = nn.Linear(w_dim, channels * 2)

    def forward(self, x, w):
        # x: (B, C, H, W), w: (B, w_dim)
        style = self.fc(w)                                  # (B, 2*C)
        scale, bias = style.chunk(2, dim=1)                 # each (B, C)
        scale = scale.unsqueeze(2).unsqueeze(3)             # (B, C, 1, 1)
        bias  = bias.unsqueeze(2).unsqueeze(3)

        # Instance norm (per-sample, per-channel)
        mean = x.mean(dim=[2, 3], keepdim=True)
        std  = x.std(dim=[2, 3], keepdim=True) + 1e-8
        normalized = (x - mean) / std

        return normalized * scale + bias

## 6. The Synthesis Block

One block in the synthesis network does:

1. (Optional) upsample 2x with nearest-neighbor
2. 3x3 convolution
3. Add per-channel-weighted noise
4. AdaIN, using `w` as the style source
5. LeakyReLU activation

The `noise_weight` is a learnable `(1, C, 1, 1)` parameter that lets the network decide how much stochastic detail each channel wants.

In [ ]:
class SynthesisBlock(nn.Module):
    def __init__(self, in_channels, out_channels, w_dim, upsample=True):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='nearest') if upsample else None
        self.conv     = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.noise_weight = nn.Parameter(torch.zeros(1, out_channels, 1, 1))
        self.adain    = AdaIN(w_dim, out_channels)
        self.act      = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x, w, noise):
        if self.upsample is not None:
            x = self.upsample(x)
        x = self.conv(x)
        x = x + self.noise_weight * noise
        x = self.adain(x, w)
        return self.act(x)

## 7. The Synthesis Network

Spatial flow for a `32x32` output:

```
learned constant 4x4 (128 ch)
  -> initial block (no upsample)        -> 4x4,   128 ch, to_rgb -> 32x32 img? no, 4x4
  -> upsample + block                   -> 8x8,    64 ch, to_rgb -> 8x8
  -> upsample + block                   -> 16x16,  32 ch, to_rgb -> 16x16
  -> upsample + block                   -> 32x32,  16 ch, to_rgb -> 32x32
```

Each block has its own AdaIN, so each block's `w` -> style mapping is *separate*. We use `w` directly (one shared style vector across all layers) for simplicity. The original StyleGAN learns a *different* affine per layer from the same `w`.

For this simplified notebook, every layer sees the same `w`. We could trivially extend by giving each layer its own linear layer from `w`.

In [ ]:
class SynthesisNetwork(nn.Module):
    def __init__(self, w_dim=128, channels=(128, 64, 32, 16)):
        super().__init__()
        self.channels = channels

        # Learned 4x4 constant (the ONLY thing not derived from z or noise)
        self.constant = nn.Parameter(torch.randn(1, channels[0], 4, 4))

        # Initial block at 4x4 (no upsample)
        self.initial_block = SynthesisBlock(
            in_channels=channels[0],
            out_channels=channels[0],
            w_dim=w_dim,
            upsample=False,
        )
        self.initial_rgb = nn.Conv2d(channels[0], img_channels, 1)

        # Subsequent blocks: upsample then convolve
        self.blocks = nn.ModuleList()
        self.to_rgbs = nn.ModuleList()
        for i in range(1, len(channels)):
            self.blocks.append(
                SynthesisBlock(
                    in_channels=channels[i-1],
                    out_channels=channels[i],
                    w_dim=w_dim,
                    upsample=True,
                )
            )
            self.to_rgbs.append(nn.Conv2d(channels[i], img_channels, 1))

    def forward(self, w, noise_list):
        """
        noise_list[i] is the (B, 1, H, W) noise tensor for level i.
        Level 0 -> 4x4, level 1 -> 8x8, level 2 -> 16x16, level 3 -> 32x32.
        """
        x = self.constant.repeat(w.size(0), 1, 1, 1)
        x = self.initial_block(x, w, noise_list[0])
        rgb = self.initial_rgb(x)

        for i, (block, to_rgb) in enumerate(zip(self.blocks, self.to_rgbs)):
            x = block(x, w, noise_list[i+1])
            rgb = to_rgb(x)

        return rgb

## 8. The Full Generator

The Generator composes the mapping network and the synthesis network. In the original StyleGAN, the synthesis network gets a *different* affine projection of `w` per layer; here we share one `w` across layers for clarity.

In [ ]:
class StyleGANGenerator(nn.Module):
    def __init__(self, latent_dim=100, w_dim=128, channels=(128, 64, 32, 16)):
        super().__init__()
        self.mapping  = MappingNetwork(latent_dim, w_dim)
        self.synthesis = SynthesisNetwork(w_dim, channels)
        self.n_levels = len(channels)

    def forward(self, z, noise=None):
        """
        z: (B, latent_dim)
        noise: optional list of (B, 1, H, W) tensors. If None, sample fresh.
        """
        w = self.mapping(z)
        if noise is None:
            noise = [
                torch.randn(z.size(0), 1, 4 * (2**i), 4 * (2**i), device=z.device)
                for i in range(self.n_levels)
            ]
        img = self.synthesis(w, noise)
        return img, w

## 9. The Discriminator

A standard PatchGAN. StyleGAN's Discriminator does not need to be clever — the architectural innovations are entirely on the Generator side. Same 70x70 receptive field idea as Pix2Pix / CycleGAN, but here with `32x32` input the output patch map is `4x4`.

In [ ]:
class PatchDiscriminator(nn.Module):
    def __init__(self, img_channels=1, features=64):
        super().__init__()
        self.net = nn.Sequential(
            # 32 -> 16, NO normalization in the first block
            nn.Conv2d(img_channels, features, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # 16 -> 8
            nn.Conv2d(features, features * 2, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(features * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # 8 -> 4
            nn.Conv2d(features * 2, features * 4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(features * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # 4 -> 4 (preserve spatial)
            nn.Conv2d(features * 4, features * 8, 3, 1, 1, bias=False),
            nn.InstanceNorm2d(features * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # final 1-channel logit map (4x4 for 32x32 input)
            nn.Conv2d(features * 8, 1, 3, 1, 1, bias=False),
        )

    def forward(self, x):
        return self.net(x)

## 10. Models, Optimizers, Loss

Same `BCEWithLogitsLoss` as Pix2Pix / CycleGAN. Adam with the standard DCGAN-style betas.

In [ ]:
G = StyleGANGenerator(latent_dim=latent_dim, w_dim=w_dim).to(device)
D = PatchDiscriminator(img_channels=img_channels, features=features_d).to(device)

G.apply(weights_init)
D.apply(weights_init)

optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=betas)
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=betas)

bce = nn.BCEWithLogitsLoss()

print(f'G params: {sum(p.numel() for p in G.parameters()):,}')
print(f'D params: {sum(p.numel() for p in D.parameters()):,}')

## 11. Fixed Noise for Visualization

Two fixed batches: a primary `z` set for the standard per-epoch grid, and a separate `z_pair` for the style-mixing demo later. Each entry of `z_pair` is paired with its neighbor so we can swap styles side-by-side.

In [ ]:
fixed_z = torch.randn(64, latent_dim, device=device)
fixed_z_pair = torch.randn(8, 2, latent_dim, device=device)   # 8 pairs of (z_a, z_b)

## 12. The Training Loop

Standard alternating pattern. The Generator's `forward` returns *both* the image and the intermediate `w`; we only need the image here, but returning `w` makes the style-mixing demo trivial later.

Patch map targets are `(B, 1, 4, 4)` for our `32x32` input.

In [ ]:
losses_g = []
losses_d = []

G.train(); D.train()

for epoch in range(epochs):
    sum_d = 0.0
    sum_g = 0.0
    n_batches = 0

    for real_imgs, _ in dataloader:
        real_imgs = real_imgs.to(device)
        b = real_imgs.size(0)

        valid  = torch.ones (b, 1, 4, 4, device=device)
        fake_t = torch.zeros(b, 1, 4, 4, device=device)

        # -----------------
        # Phase A: Train D
        # -----------------
        optimizer_D.zero_grad()

        # Real
        real_logits = D(real_imgs)
        d_real_loss = bce(real_logits, valid)

        # Fake (detached)
        z = torch.randn(b, latent_dim, device=device)
        with torch.no_grad():
            gen_imgs, _ = G(z)
        fake_logits = D(gen_imgs)
        d_fake_loss = bce(fake_logits, fake_t)

        d_loss = (d_real_loss + d_fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()

        # -----------------
        # Phase B: Train G
        # -----------------
        optimizer_G.zero_grad()

        gen_imgs, _ = G(z)                       # NO detach
        validity_logits = D(gen_imgs)
        g_loss = bce(validity_logits, valid)
        g_loss.backward()
        optimizer_G.step()

        sum_d += d_loss.item()
        sum_g += g_loss.item()
        n_batches += 1

    avg_d = sum_d / n_batches
    avg_g = sum_g / n_batches
    losses_d.append(avg_d)
    losses_g.append(avg_g)

    print(f"Epoch [{epoch+1:2d}/{epochs}]  D: {avg_d:.3f}  G: {avg_g:.3f}")

    G.eval()
    with torch.no_grad():
        sample_imgs, _ = G(fixed_z)
    save_image(
        sample_imgs.detach().cpu(),
        f"samples_StyleGAN/epoch_{epoch+1:02d}.png",
        nrow=8,
        normalize=True,
    )
    G.train()

print('Training done.')

## 13. Random Samples at 32x32

Final visualization. Each cell is a sample from the trained Generator.

In [ ]:
G.eval()
with torch.no_grad():
    samples, _ = G(fixed_z)

grid = make_grid(samples.cpu(), nrow=8, normalize=True)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
plt.axis('off')
plt.title('StyleGAN — random samples at 32x32')
plt.show()

## 14. Style Mixing — The Signature StyleGAN Demo

StyleGAN's whole point is that different layers can absorb different aspects of the style. We can demonstrate this by feeding `w_A` into the early layers and `w_B` into the late layers — the resulting image should look like `B`'s fine details but inherit some coarse structure from `A`.

Since our simplified StyleGAN shares one `w` across all layers, we instead do the *original* style-mixing trick: pass `z_A` to the synthesis network for the **early** noise levels and `z_B` for the **late** ones.

For each pair `(z_A, z_B)` we generate:

1. `A` — generated from `z_A` alone.
2. `B` — generated from `z_B` alone.
3. `mix` — early noise from `z_A`, late noise from `z_B`.

If the model has learned to use noise at different layers for different purposes, the mix should resemble `B` overall but inherit some structural randomness from `A`.

In [ ]:
G.eval()

with torch.no_grad():
    z_a = fixed_z_pair[:, 0, :]   # (8, latent_dim)
    z_b = fixed_z_pair[:, 1, :]   # (8, latent_dim)

    # Full A and full B images
    img_a, _ = G(z_a)
    img_b, _ = G(z_b)

    # Style mixing: noise from A in the first 2 levels, noise from B in the last 2
    noise_a = [
        torch.randn(8, 1, 4 * (2**i), 4 * (2**i), device=device)
        for i in range(G.n_levels)
    ]
    noise_b = [
        torch.randn(8, 1, 4 * (2**i), 4 * (2**i), device=device)
        for i in range(G.n_levels)
    ]
    noise_mix = noise_a[:2] + noise_b[2:]

    img_mix, _ = G(z_b, noise=noise_mix)   # use z_b but mix noise

# Stack: row 1 = A, row 2 = B, row 3 = mix
comparison = torch.cat([img_a.cpu(), img_b.cpu(), img_mix.cpu()], dim=0)
grid = make_grid(comparison, nrow=8, normalize=True)
plt.figure(figsize=(10, 5))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
plt.axis('off')
plt.title('Style mixing demo — top: A | middle: B | bottom: B with A\'s early noise')
plt.show()

## 15. Training Progression

Each saved epoch grid stacked left-to-right. Watch the Generator transition from "blurry averages" to recognizable digits.

In [ ]:
from PIL import Image
import glob

paths = sorted(glob.glob('samples_StyleGAN/epoch_*.png'))
if paths:
    fig, axes = plt.subplots(1, len(paths), figsize=(1.4 * len(paths), 1.4))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        ax.imshow(np.array(Image.open(p)).squeeze(), cmap='gray')
        ax.set_title(p.split('_')[-1].split('.')[0], fontsize=7)
        ax.axis('off')
    plt.suptitle('StyleGAN — same fixed z, sampled every epoch')
    plt.show()
else:
    print('No sample grids found. Run the training cell first.')

## 16. Loss Curves

Standard adversarial BCE dynamics. Watch for D approaching `ln(2) ~ 0.693` and G tracking it.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, epochs + 1), losses_d, label='D', color='tab:blue')
plt.plot(range(1, epochs + 1), losses_g, label='G', color='tab:orange')
plt.axhline(np.log(2), color='gray', linestyle='--', alpha=0.5, label='ln 2')
plt.xlabel('Epoch')
plt.ylabel('BCELoss')
plt.title('StyleGAN losses')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Recap — How StyleGAN Differs From Earlier Architectures

| Concern | PGAN / Pix2Pix / CycleGAN | StyleGAN |
|---|---|---|
| Latent input | `z` consumed at the start | **`z` mapped to `w` first**, then injected at every layer |
| Initial feature | depends on `z` | **learned constant** |
| Style injection | none | **AdaIN** at every synthesis layer |
| Stochastic variation | implicit in `z` | **per-layer noise injection** separate from `w` |
| Hierarchical control | not by construction | **early layers -> coarse, late layers -> fine** |
| Style mixing | n/a | **enabled by construction** — swap `w` slices between images |
| Normalization | varies | **InstanceNorm inside AdaIN** |
| Loss | varies | **BCE** |
| Progressive growing | used in PGAN | optional (the original StyleGAN combines both) |

**What changed and why.**

- **Mapping network.** A small MLP transforms `z` into `w`. Empirically `w` ends up smoother and more disentangled than `z` — moving along a single direction in `w` tends to change a single semantic attribute, whereas moving along a direction in `z` tends to change several things at once.
- **Learned constant input.** The synthesis network does not consume `z` directly. The first feature map is a *learned* tensor that gets modulated by styles. This forces all variation in the output to come from the style inputs — there is no "noise enters here" shortcut.
- **AdaIN.** Normalize the activations, then replace their per-channel mean and standard deviation with values derived from `w`. This is a strong inductive bias: each layer's style lives in a *tiny* linear projection of `w`, so the model has to learn *what each layer should pay attention to* in the style vector.
- **Per-layer noise.** Without separate noise channels, all fine-scale stochastic detail would have to be encoded into `w`. By giving each layer its own learnable noise weight, the network can devote `w` to *structured* style attributes and reserve noise for *unstructured* fine detail.
- **Style mixing works because of layering.** Because different synthesis layers absorb different parts of the style space, you can mix two latents' styles at different layer boundaries and get coherent combinations. This is *emergent* in the architecture; you don't have to engineer it.

**Common implementation pitfalls** — quick reference:

- **Noise broadcast.** The noise input has shape `(B, 1, H, W)` — a single channel, broadcast across all feature channels. The learnable `noise_weight` of shape `(1, C, 1, 1)` scales it per channel. Don't pass `(B, C, H, W)` noise — the broadcast breaks and the network can't learn the per-channel weights cleanly.
- **AdaIN affine size.** The linear layer in AdaIN outputs `2 * C` (scale + bias). A common mistake is to output only `C`, which silently leaves the bias unlearned.
- **Style mixing needs *different* noise per layer.** Reusing the same noise tensor across all layers in the mix defeats the purpose. Make sure `noise_a` and `noise_b` are independent samples.
- **The mapping network is small but important.** Four linear layers is *not* a lot of capacity, but it does the heavy lifting of disentangling. Skipping it (using `z` directly in the synthesis network) collapses back to ordinary GANs.
- **The learned constant is *the only* learnable thing that isn't derived from `z` or `w`.** It converges to a specific pattern during training. Don't initialize it from noise at every forward pass — it's a parameter, not an input.
- **StyleGAN2 is *not* just StyleGAN with better hyperparameters.** It replaces AdaIN with weight modulation/demodulation specifically because of an artifact AdaIN produces. Module 19 covers that redesign.

**Why StyleGAN was a major shift.** StyleGAN reframed the latent vector from *random input noise* into *a control signal for the synthesis process*. Once you can inject the same `w` at every layer, the layers specialize, the latent space becomes more meaningful, and downstream tasks — interpolation, style mixing, attribute editing — fall out naturally. It's the bridge from "generate realistic images" to "generate realistic images I can control."

**Next in the series** (Module 19): StyleGAN2. The original StyleGAN has a known artifact (droplet-like blobs) caused by how AdaIN interacts with feature statistics. StyleGAN2 fixes it by replacing AdaIN with **weight modulation** and **weight demodulation**, removes progressive growing, and adds **path length regularization**. Cleaner images, simpler architecture, faster training.